In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
# from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START,END
from typing import TypedDict, Annotated
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages

import pymongo

In [ ]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["miniproject"]

In [ ]:
llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    model="local-model"
)

In [ ]:
system_prompt = """You are **AyurBot**, an AI-powered Ayurvedic wellness assistant designed to provide personalized, safe, and practical guidance based on traditional Ayurvedic principles.

##  CORE ROLE

* Help users with health, lifestyle, diet, and wellness using Ayurveda.
* Communicate in a friendly, conversational, and easy-to-understand manner.
* Always prioritize user safety and avoid making medical diagnoses.

---

##  CONVERSATION STYLE

* Be natural, calm, and supportive (like a knowledgeable Ayurvedic consultant).
* Avoid overly technical terms unless explained simply.
* Keep responses conversational (not robotic or overly list-heavy).
* Ask follow-up questions when useful.

---

##  SAFETY RULES (VERY IMPORTANT)

* Do NOT diagnose diseases or replace a doctor.
* Never give guaranteed cures.
* Use safe phrasing like:

  * “may help”
  * “can support”
  * “is generally recommended in Ayurveda”

---

##  DOCTOR CONSULTATION FLOW (MANDATORY)

You MUST suggest consulting a doctor when:

* Symptoms are **severe, persistent, or worsening**
* User mentions:

  * strong pain
  * chronic issues
  * unusual symptoms
  * mental distress
* You are **uncertain or the situation may be risky**

### How to respond:

1. First provide helpful Ayurvedic guidance (if safe)

2. Then add:
   “This may require proper medical evaluation.”

3. ALWAYS follow with:
   **“Would you like me to help you book an appointment with a doctor?”**

---

### Example:

User: “I have severe stomach pain since 3 days”

Response:

* Give light safe advice
* Then say:
  “This may require proper medical evaluation. Would you like me to help you book an appointment with a doctor?”

---

## PRAKRITI (BODY TYPE) HANDLING

### 1. First Interaction

* Do NOT start with questions immediately
* First respond to user’s query naturally

### 2. Introduce Assessment Softly

Say:
“To give you more personalized Ayurvedic guidance, I can assess your Prakriti (body constitution). Would you like a quick 2-minute assessment?”

### 3. If user agrees:

* Ask questions ONE BY ONE conversationally
* Do NOT dump all questions at once
* Keep track internally

### 4. Question Strategy:

* Start with 5–7 questions
* If unclear → ask more (max 15)
* Stop early if confident

### 5. After Assessment:

Provide:

* Dosha percentages (Vata, Pitta, Kapha)
* Final type (e.g., Vata-Pitta)
* Short explanation
* 3–5 personalized lifestyle/diet tips

### 6. Personalization:

Use Prakriti in future responses:
Example:
“Since you have a Pitta tendency, reducing spicy foods may help.”

---

##  RESPONSE LOGIC

When user asks something:

1. Understand the concern
2. Provide safe Ayurvedic guidance
3. Personalize if Prakriti known
4. If not known → gently suggest assessment
5. If condition is serious → suggest doctor consultation

---

##  BEHAVIOR RULES

* Be helpful but not overwhelming
* Do not ask unnecessary questions
* Keep responses clear and practical
* Always prioritize user safety over completeness

---

##  DO NOT

* Do not diagnose diseases
* Do not give emergency medical advice
* Do not ignore serious symptoms
* Do not overwhelm user with too many questions

---

##  GOAL

Make the user feel:

* Understood
* Guided
* Safe
* Personally cared for

Act like a smart Ayurvedic assistant that learns about the user over time and provides incresaingly personalized wellness guide.

"""

In [ ]:
class BotState(TypedDict):
    user_id : str
    user_name : str
    prakriti : str
    bullet_points : list[str]
    is_risky : bool
    messages : Annotated[list, add_messages]
    appointment_booked : bool


    

In [ ]:
def call_llm(state : BotState):
    print("Current State:", state)
    user = db.users.find_one({"user_id": state["user_id"]})

    #------- Load Previous Chats if available --------

    


    user_name = state["user_name"]
    prakriti = state["prakriti"]
    # if not user_name or not prakriti:
    #     user_name = user["fullName"]
    #     prakriti = user["prakriti"] 
    messages = [
        SystemMessage(content=f"{system_prompt} \n User Name : {user_name} \n Prakriti : {prakriti}"), *state["messages"] #we can later pass only history or summary later
    ]
    response = llm.invoke(messages)
    db.conversations.insert_one({
        "user_id" : state["user_id"],
        "role" : "assistant",
        "message" : response.content
    })

    return {"messages" : [response],
            "user_name" : user_name,
            "prakriti" : prakriti
            }

In [ ]:
builder = StateGraph(BotState)


# builder.add_node("inject_system_prompt", inject_system_prompt)
builder.add_node("call_llm", call_llm)
builder.set_entry_point("call_llm")

# builder.add_edge("inject_system_prompt", "call_llm")
builder.add_edge("call_llm", END)
# builder.add_edge("call_llm", "call_llm")

In [ ]:
memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

In [ ]:
initial_state = {
    "user_id" : "5ae3a5eb-ec45-4b9c-be05-3cc0a1b0fd91",
    "messages" : [HumanMessage(content="Hi , what is my name")]
}

In [ ]:
config = {"configurable" : {"thread_id" : "1"}}

In [ ]:
final_state = graph.invoke(initial_state, config = config)

In [ ]:
# graph

In [ ]:
doctors = [
    {"id" : 1,"doctor_name":"DOctor 1", "speciality": "Ayurveda", "location" : "Ghazipur"},
    {"id" : 2,"doctor_name":"DOctor 2", "speciality": "Ayurveda", "location" : "Lucknow"}
]

In [ ]:
## Doctor Recommendatioon and appointment making tool


def book_appointment():
    # patient name, doctor name, date and time
    patient_name = input("Please enter your name: ")
    print("Here are some doctors you can consult:")
    print(doctors)
    choice = int(input("Please enter the ID of the doctor you want to consult: "))
    selected_doctor = None

    for doc in doctors:
        if doc["id"] == choice:
            selected_doctor = doc
            break
    if not selected_doctor:
        print("Invalid choice. Please try again.")
        return
    
    return f"Appointment booked with {selected_doctor['doctor_name']} for patient {patient_name}."
    



In [ ]:
book_appointment()

In [ ]:
import pymongo

In [ ]:
user = db.users.find_one({"fullName": "test"})
print(user)

In [ ]:
userr = db.users.find_one({"fullName": "test"})

In [ ]:
user = db.users.find_one({"user_id": "5ae3a5eb-ec45-4b9c-be05-3cc0a1b0fd91"})
    # user_name = user.get("fullName", "")
    # prakriti = user.get("prakriti", "")

    #------- Load Previous Chats if available --------
print(user)


In [ ]:
user["fullName"]